# Stellar mass from the aligned image--spectrum embedding

Table 1 (aligned vs unaligned at $1<z\le3$), the modality decomposition, and the
redshift-binned figure. Writes `mass.png`.

In [ ]:
import sys, glob, re
import numpy as np, h5py, torch
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import ConnectionPatch
from astropy.table import Table
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

from plotstyle import (ROOT, FUSION_DIR, EMBED_H5, DJA_FITS, use_style, style_axes, fs, save)
sys.path.insert(0, str(FUSION_DIR))
from model.fusion import MultimodalFusion

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CKPT = max(glob.glob(str(FUSION_DIR / "outputs/fusion_dja_image_attn_spectrum_attn"
                          / "version_*/checkpoints/*R10=*.ckpt")),
           key=lambda p: float(re.search(r"R10=([0-9.]+)", p).group(1)))

FRAC_VALID_PIX, MIN_SN50 = 0.5, 0        # spectrum window coverage / median S/N
MAX_DZPHOT               = 0.15          # |z_phot - z_best|/(1+z): keep masses fitted at the right redshift
ZLO, ZHI                 = 1.0, 3.0      # table window
MAXZ_FIG                 = 7.0           # figure window
N_SPLITS                 = 40
ALPHAS                   = np.logspace(-2, 6, 25)
SEED                     = 42

use_style(1.5)
print(f"device={DEVICE}\nckpt={CKPT.split('/')[-1]}")

## 1. Sample

In [ ]:
# === sample ===================================================================
# Everything below is read once; `sel_fig` is the redshift-binned figure sample and
# `zi` indexes the 1<z<=3 subset used for the table.
dja = Table.read(DJA_FITS)
with h5py.File(EMBED_H5, "r") as f:
    ids  = f["id"][:]
    surv = f["survey"][:].astype(str)
    sn   = f["sn50"][:]
    smask = f["spectrum_token_mask"][:].astype(bool)
    spec  = f["spectrum_patch_embed"][:]
    icls  = f["image_cls_embed"][:]
    istat = f["image_stats"][:].astype(np.float32)
    sstat = f["spectrum_stats"][:].astype(np.float32)
    spec_ok = np.isfinite(spec.reshape(len(ids), -1)).all(1)

with np.errstate(invalid="ignore", divide="ignore"):
    logm = np.log10(np.asarray(dja["phot_mass"])[ids].astype(np.float32))
z    = np.asarray(dja["z_best"])[ids].astype(np.float32)
zph  = np.asarray(dja["z_phot"])[ids].astype(np.float32)
f150 = np.asarray(dja["phot_f150w_tot_1"])[ids].astype(np.float64)

# phot_mass is an EAZY fit to the photometry, so it is only usable where the
# photometric redshift agrees with the spectroscopic one.
with np.errstate(invalid="ignore", divide="ignore"):
    dzp = np.abs((zph - z) / (1 + z))

# the cross-matched sample of Section 2 is built from the grade-filtered spectra,
# so the same cut is applied here to keep the sample chain consistent.
sys.path.insert(0, str(ROOT / "encoder_spectrum/LowResPT"))
from data.dataset import LowResDataset
_keep = np.asarray(LowResDataset(fits_path=str(DJA_FITS), grades=(1, 2, 3),
                                 frac_valid_pix=0.5).valid_indices)

sel_fig = (spec_ok & np.isfinite(logm) & np.isfinite(z) & (z > 0)
           & (smask.mean(1) > FRAC_VALID_PIX) & (sn > MIN_SN50)
           & np.isin(ids, _keep)
           & np.isfinite(dzp) & (dzp < MAX_DZPHOT) & (z <= MAXZ_FIG))
rows = np.where(sel_fig)[0]

# image patch tokens are the only large array: read just the selected rows
with h5py.File(EMBED_H5, "r") as f:
    d = f["image_patch_embed"]
    imgp = np.empty((len(rows), d.shape[1], d.shape[2]), np.float32)
    for s in range(0, len(rows), 512):
        imgp[s:s + 512] = d[rows[s:s + 512]]

spec, smask, icls = spec[rows], smask[rows], icls[rows]
istat, sstat      = istat[rows], sstat[rows]
y, zz, surv, ids  = logm[rows], z[rows], surv[rows], ids[rows]
lf                = np.log10(f150[rows])
print(f"figure sample (z<={MAXZ_FIG}): {len(rows)}   image tokens {imgp.shape}")

## 2. Frozen CLIP latents

In [ ]:
# === frozen CLIP latents ======================================================
ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
hp = ck["hyper_parameters"]
fusion = MultimodalFusion(latent_dim=hp["latent_dim"], temperature=hp["temperature"])
for name, cfg in hp["modalities"].items():
    fusion.register_modality(name, input_dim={"image": imgp.shape[-1], "spectrum": spec.shape[-1]}[name],
                             hidden_dim=cfg.get("hidden_dim"), pool=cfg.get("pool"),
                             num_heads=cfg.get("num_heads", 4), stats_dim=cfg.get("stats_dim"))
fusion.load_state_dict({k[len("fusion."):]: v for k, v in ck["state_dict"].items()
                        if k.startswith("fusion.")}, strict=True)
fusion = fusion.to(DEVICE).eval()

@torch.no_grad()
def clip_latents(bs=256):
    raw = {"image": istat, "spectrum": sstat}
    ei, es = [], []
    for s in range(0, len(y), bs):
        sl = slice(s, s + bs)
        im = torch.from_numpy(imgp[sl]).to(DEVICE); sp = torch.from_numpy(spec[sl]).to(DEVICE)
        av = {n: torch.ones(im.shape[0], dtype=torch.bool, device=DEVICE) for n in ("image", "spectrum")}
        st = {m: torch.from_numpy(raw[m][sl]).to(DEVICE) for m in fusion.stats_encoders}
        e = fusion({"image": im, "spectrum": sp}, av,
                   {"spectrum": torch.from_numpy(smask[sl]).to(DEVICE)}, st or None)
        ei.append(e["image"].cpu().numpy()); es.append(e["spectrum"].cpu().numpy())
    return np.concatenate(ei), np.concatenate(es)

IMG_AL_ALL, SPC_AL_ALL = clip_latents()
print("aligned latents:", IMG_AL_ALL.shape, SPC_AL_ALL.shape)

## 3. Representations

Unaligned readouts pool the frozen tokens with no learned parameters, at a width
close to the 256-d aligned embedding.

In [ ]:
# === representations ==========================================================
# The unaligned readouts take the frozen encoder output with NO learned parameters, and
# are kept close in width to the 256-d aligned embedding so the comparison is not
# confounded by dimensionality: image = [CLS], spectrum = masked mean + masked max.
# Both append the raw (mean, std) flux statistics, which the aligned head also receives.
w      = smask[..., None].astype(np.float32)
smean  = (spec * w).sum(1) / np.clip(w.sum(1), 1, None)
smax   = np.nan_to_num(np.where(smask[..., None], spec, -np.inf).max(1), neginf=0.0)
ist, sst = np.arcsinh(istat), np.arcsinh(sstat)

IMG_UN_ALL = np.hstack([icls, ist])                    # [CLS] + stats
SPC_UN_ALL = np.hstack([smean, smax, sst])             # masked mean + max + stats
print(f"image un. {IMG_UN_ALL.shape[1]}d | spectrum un. {SPC_UN_ALL.shape[1]}d | "
      f"aligned {IMG_AL_ALL.shape[1]}d each")

zi = np.where((zz > ZLO) & (zz <= ZHI) & np.isfinite(lf))[0]     # table sample
PHOT     = lf[zi][:, None]
IMG_UN, SPC_UN = IMG_UN_ALL[zi], SPC_UN_ALL[zi]
IMG_AL, SPC_AL = IMG_AL_ALL[zi], SPC_AL_ALL[zi]
JNT_UN, JNT_AL = np.hstack([IMG_UN, SPC_UN]), np.hstack([IMG_AL, SPC_AL])
yz, zt, lft = y[zi], zz[zi], lf[zi]
splits = [train_test_split(np.arange(len(zi)), test_size=0.5, random_state=s)
          for s in range(N_SPLITS)]
print(f"table sample {ZLO}<z<={ZHI}: n={len(zi)} ({len(np.unique(ids[zi]))} unique spectra)")

## 4. Table 1 --- aligned vs unaligned at $1<z\le3$

In [ ]:
# === Table 1: aligned vs unaligned at 1 < z <= 3 ==============================
def probe(X, t, tr, va):
    sc = StandardScaler().fit(X[tr])
    return RidgeCV(alphas=ALPHAS).fit(sc.transform(X[tr]), t[tr]).predict(sc.transform(X[va]))

def metrics(X, t=None):
    t = yz if t is None else t
    out = []
    for tr, va in splits:
        p = probe(X, t, tr, va); e = p - t[va]
        out.append((r2_score(t[va], p),
                    1.4826 * np.median(np.abs(e - np.median(e))),
                    float(np.mean(np.abs(e) > 0.5))))
    return np.array(out)

ROWS = [("F150W photometry", PHOT, None),
        ("Image",    IMG_UN, IMG_AL),
        ("Spectrum", SPC_UN, SPC_AL),
        ("Image + spectrum", JNT_UN, JNT_AL)]
M = {}
print(f"1 < z <= 3, n={len(zi)}, {N_SPLITS} paired 50/50 splits (+/- = split-to-split scatter)\n")
print(f"{'representation':18s} {'':4s} {'R2':>16s} {'sigma_NMAD':>16s} {'f>0.5':>15s}")
print("-" * 74)
for name, Xu, Xa in ROWS:
    for tag, X in (("un.", Xu), ("al.", Xa)):
        if X is None: continue
        v = metrics(X); M[(name, tag)] = v
        print(f"{name:18s} {tag:4s} {v[:,0].mean():7.3f}+/-{v[:,0].std():5.3f} "
              f"{v[:,1].mean():7.3f}+/-{v[:,1].std():5.3f} "
              f"{100*v[:,2].mean():6.1f}+/-{100*v[:,2].std():4.1f}%")

print("\npaired effect of alignment (same splits):")
for name in ("Image", "Spectrum", "Image + spectrum"):
    d  = M[(name, "al.")][:, 0] - M[(name, "un.")][:, 0]
    dn = M[(name, "al.")][:, 1] - M[(name, "un.")][:, 1]
    print(f"  {name:18s} dR2 = {d.mean():+.4f} +/- {d.std(ddof=1):.4f}   "
          f"d sigma_NMAD = {dn.mean():+.4f} +/- {dn.std(ddof=1):.4f}")

## 5. What alignment moves

The same representations probed for redshift, brightness and mass.

In [ ]:
# === what alignment moves: redshift, brightness, mass from the same features ===
# Mass needs a distance and a luminosity scale. Redshift is constrained only by the spectrum;
# the total F150W flux is measured cleanly only by the image. Probing all three targets from
# the SAME representations shows what each modality was missing and what alignment supplies.
TARGETS = [("redshift z", zt), ("log F150W", lft), ("log M*", yz)]
GRID = [("image    un.", IMG_UN), ("image    al.", IMG_AL),
        ("spectrum un.", SPC_UN), ("spectrum al.", SPC_AL),
        ("joint    un.", JNT_UN), ("joint    al.", JNT_AL)]
G = {}
hdr = f"{'representation':15s} {'dim':>6s} " + " ".join(f"{n:>16s}" for n, _ in TARGETS)
print(hdr); print("-" * len(hdr))
for name, X in GRID:
    row = []
    for tn, t in TARGETS:
        v = np.array([r2_score(t[va], probe(X, t, tr, va)) for tr, va in splits])
        G[(name, tn)] = v
        row.append(f"{v.mean():7.3f}+/-{v.std():5.3f}")
    print(f"{name:15s} {X.shape[1]:6d} " + " ".join(row))

print("\npaired al. - un.:")
for m in ("image   ", "spectrum", "joint   "):
    s = f"  {m}"
    for tn, _ in TARGETS:
        d = G[(f"{m} al.", tn)] - G[(f"{m} un.", tn)]
        s += f"   {tn}={d.mean():+.3f}+/-{d.std(ddof=1):.3f}"
    print(s)

print("\nreading: each modality gains ONLY on the quantity it was weak at, and that quantity is")
print("the other modality's strength. The joint readout already reads each quantity at the")
print("better of the two single modalities, so alignment has nothing left to surface.")

## 6. Modality contributions

The joint prediction split by block, $\hat y = c + a_{\rm img} + a_{\rm spec}$.

In [ ]:
# === how much of the joint prediction comes from each modality ================
# The probe is linear, yhat = c + w_img.x_img + w_spec.x_spec, so the prediction
# splits exactly into a per-modality term. Each term's contribution is
#   u_m = Cov(w_m.x_m, y) / Var(y),
# measured on the held-out half. Covariance is additive, so u_img + u_spec equals
# Cov(y, yhat)/Var(y) exactly; that equals the joint R^2 only for a perfectly
# calibrated prediction (unbiased, unit regression slope on y), so both are shown.
#
# The second table is the coefficient view. Features are centred, so each block's
# prediction term has zero mean by construction and the whole mean level of the
# target sits in the intercept, which equals mean(y_train): a block has no "mean
# contribution" to attribute. mean(w) likewise cancels to numerical noise, orders
# of magnitude below |w|. Only |w| is meaningful, and only at equal block width.
def contrib(Xi, Xs):
    X, di = np.hstack([Xi, Xs]), Xi.shape[1]
    out = []
    for tr, va in splits:
        sc = StandardScaler().fit(X[tr])
        Xtr, Xva = sc.transform(X[tr]), sc.transform(X[va])
        m = RidgeCV(alphas=ALPHAS).fit(Xtr, yz[tr])
        w, t = m.coef_, yz[va]
        out.append([np.cov(t, Xva[:, :di] @ w[:di])[0, 1] / np.var(t),
                    np.cov(t, Xva[:, di:] @ w[di:])[0, 1] / np.var(t),
                    r2_score(t, m.predict(Xva)),
                    w[:di].mean(), w[di:].mean(),
                    np.linalg.norm(w[:di]), np.linalg.norm(w[di:]),
                    m.intercept_, yz[tr].mean(), (Xtr[:, :di] @ w[:di]).mean()])
    return np.array(out)

C = {tag: contrib(Xi, Xs) for tag, Xi, Xs in
     [("un.", IMG_UN, SPC_UN), ("al.", IMG_AL, SPC_AL)]}

print(f"modality contributions to the joint mass probe, {ZLO}<z<={ZHI}, n={len(zi)}\n")
print(f"{'':6s} {'u_image':>15s} {'u_spectrum':>15s} {'sum':>8s} {'R2':>8s} "
      f"{'share image':>15s} {'share spectrum':>16s}")
print("-" * 90)
for tag, v in C.items():
    m, s = v.mean(0), v.std(0)
    f = 100 * v[:, :2] / v[:, :2].sum(1, keepdims=True)
    print(f"{tag:6s} {m[0]:7.3f}+/-{s[0]:5.3f} {m[1]:7.3f}+/-{s[1]:5.3f} "
          f"{m[0]+m[1]:8.3f} {m[2]:8.3f} "
          f"{f[:,0].mean():9.1f}+/-{f[:,0].std():3.1f}% "
          f"{f[:,1].mean():10.1f}+/-{f[:,1].std():3.1f}%")

print("\ncoefficient view (|w| comparable across blocks only at equal width, i.e. al.)")
print(f"{'':6s} {'mean(w_img)':>12s} {'mean(w_spc)':>12s} {'|w_img|':>9s} {'|w_spc|':>9s} "
      f"{'intercept':>10s} {'mean(y_train)':>14s} {'mean(a_img)':>12s}")
print("-" * 92)
for tag, v in C.items():
    m = v.mean(0)
    print(f"{tag:6s} {m[3]:12.2e} {m[4]:12.2e} {m[5]:9.3f} {m[6]:9.3f} "
          f"{m[7]:10.4f} {m[8]:14.4f} {m[9]:12.2e}")

## 7. Redshift-binned figure

In [ ]:
# === paper figure: stellar mass in redshift bins =============================
# One ridge probe is trained once on the training half of the full z<=7 sample;
# only its test-half predictions are then split into the five redshift bins.

EDGES = np.array([0.5, 1.5, 2.5, 3.5, 4.5, 7.0])          # five evaluation bins
NB2   = len(EDGES) - 1
BIN_MIN_N = 10
SURVEY_PALETTE = {"cosmos": "#0072B2", "ceers": "#E69F00",
                  "jades": "#CC79A7", "outthere": "#009E73"}
SVORDER = [s for s in ("jades", "cosmos", "ceers", "outthere") if np.any(surv == s)]
BCOL = plt.cm.viridis(np.linspace(0.15, 0.85, NB2))

tr_all, va_all = train_test_split(np.arange(len(y)), test_size=0.5, random_state=SEED)
CLIP_ALL = np.hstack([IMG_AL_ALL, SPC_AL_ALL])
_sc  = StandardScaler().fit(CLIP_ALL[tr_all])
_reg = RidgeCV(alphas=ALPHAS).fit(_sc.transform(CLIP_ALL[tr_all]), y[tr_all])
rpred = _reg.predict(_sc.transform(CLIP_ALL[va_all]))
yva, zva, survey_va = y[va_all], zz[va_all], surv[va_all]
_e = rpred - yva
print("RIDGE CLIP alpha=%-8.3g R2=%.3f  sigmaNMAD=%.3f  out=%.1f%%"
      % (_reg.alpha_, r2_score(yva, rpred),
         1.4826 * np.median(np.abs(_e - np.median(_e))), 100 * np.mean(np.abs(_e) > 0.5)))

def _bin_mask2(j):
    m = (zva > EDGES[j]) & (zva <= EDGES[j + 1])
    return m if m.sum() >= BIN_MIN_N else None

# per-bin numbers are the mean over N_SPLITS random halves; the scatter drawn is
# the single SEED split, so every point is a real object shown once.
def _m(t, p):
    d = p - t
    return (r2_score(t, p), 1.4826 * np.median(np.abs(d - np.median(d))),
            float(np.mean(np.abs(d) > 0.5)))

_acc = {j: [] for j in list(range(NB2)) + ["all"]}
for _s in range(N_SPLITS):
    _tr, _va = train_test_split(np.arange(len(y)), test_size=0.5, random_state=_s)
    _scs = StandardScaler().fit(CLIP_ALL[_tr])
    _p = (RidgeCV(alphas=ALPHAS).fit(_scs.transform(CLIP_ALL[_tr]), y[_tr])
          .predict(_scs.transform(CLIP_ALL[_va])))
    _acc["all"].append(_m(y[_va], _p) + (len(_va),))
    for j in range(NB2):
        mk = (zz[_va] > EDGES[j]) & (zz[_va] <= EDGES[j + 1])
        if mk.sum() >= BIN_MIN_N:
            _acc[j].append(_m(y[_va][mk], _p[mk]) + (int(mk.sum()),))
BIN = {k: (np.array(v).mean(0), np.array(v).std(0)) for k, v in _acc.items() if v}

print(f"\nper-bin over {N_SPLITS} halves (mean +/- split-to-split scatter)")
for k in list(range(NB2)) + ["all"]:
    if k not in BIN: continue
    m, sd = BIN[k]
    lab = "all" if k == "all" else f"({EDGES[k]:.1f},{EDGES[k+1]:.1f}]"
    print(f"{lab:14s} <n>={m[3]:5.0f}  R2={m[0]:.3f}+/-{sd[0]:.3f}  "
          f"sNMAD={m[1]:.3f}+/-{sd[1]:.3f}  f>0.5={100*m[2]:.1f}+/-{100*sd[2]:.1f}%")
print("caption: scatter at most %.3f in R2, %.3f dex in sigma_NMAD, %.1f per cent in f>0.5"
      % (max(BIN[k][1][0] for k in BIN), max(BIN[k][1][1] for k in BIN),
         100 * max(BIN[k][1][2] for k in BIN)))

_lo = max(np.concatenate([yva, rpred]).min() - 0.2,
          np.percentile(np.concatenate([yva, rpred]), 1) - 0.3)
_hi = np.concatenate([yva, rpred]).max() + 0.2
TICKS = [t for t in range(4, 15) if _lo < t < _hi]
while TICKS[-1] < 12:
    TICKS.append(TICKS[-1] + 1)
GLIM2 = (TICKS[0] - 0.9, TICKS[-1] + 0.5)
# extra headroom above the top tick, so the per-panel annotation clears the points
YLIM2 = (GLIM2[0], TICKS[-1] + 1.0)

fig = plt.figure(figsize=(15, 4.4))
outer = gridspec.GridSpec(2, 1, height_ratios=[0.20, 1.0], hspace=0.10)
gs = outer[1].subgridspec(1, NB2, hspace=0.05, wspace=0.03)

axz = fig.add_subplot(outer[0])
zbins = np.linspace(EDGES[0], EDGES[-1], max(40, NB2 * 8) + 1)
axz.hist(zz, bins=zbins, color="0.75", edgecolor="white", label="all")
axz.hist(zz[tr_all], bins=zbins, histtype="step", color="k", lw=1.6, label="train")
for j in range(NB2):
    axz.axvspan(EDGES[j], EDGES[j + 1], color=BCOL[j], alpha=0.22)
for e in EDGES:
    axz.axvline(e, color="k", lw=1.0, ls="--", alpha=0.7)
axz.set_ylabel("count", fontsize=fs(12), labelpad=2)
axz.yaxis.set_major_locator(mpl.ticker.MaxNLocator(2, prune="lower"))
axz.set_xlim(EDGES[0], EDGES[-1])
style_axes(axz)
axz.xaxis.set_ticks_position('top'); axz.xaxis.set_label_position('top')
axz.tick_params(axis='x', labeltop=True, labelbottom=False, labelsize=fs(14))
axz.set_xlabel("redshift $z$", fontsize=fs(13.5), labelpad=2)
axz.legend(fontsize=fs(7), loc="upper right", frameon=False, handlelength=1.2,
           handletextpad=0.4, borderaxespad=0.2, bbox_to_anchor=(1.0, 0.98))

top_panels, row_first = [], None
for j in range(NB2):
    ax = fig.add_subplot(gs[0, j], sharey=row_first)
    if row_first is None:
        row_first = ax
    ax.set_box_aspect(1); ax.grid(False)
    top_panels.append(ax)
    mk = _bin_mask2(j)
    if mk is not None:
        yv, pv, sv = yva[mk], rpred[mk], survey_va[mk]
        for s in SVORDER:
            m2 = sv == s
            if m2.any():
                ax.scatter(yv[m2], pv[m2], s=13, alpha=0.3, color=SURVEY_PALETTE[s],
                           edgecolors="none", rasterized=True, label=s if j == 0 else None)
        ax.plot(GLIM2, GLIM2, "--", color="0.35", lw=1.2, alpha=0.75, dashes=(5, 3))
        d = pv - yv
        ax.text(0.96, 0.05,
                rf"$R^2$={BIN[j][0][0]:.2f}" + "\n"
                + rf"$\sigma_{{\rm NMAD}}$={BIN[j][0][1]:.2f}",
                transform=ax.transAxes, ha='right', va='bottom', fontsize=fs(8),
                bbox=dict(facecolor='white', alpha=0.7, pad=2.0, edgecolor='none'))
        ax.text(0.05, 0.95, f"n={len(yv)}", transform=ax.transAxes, ha='left',
                va='top', fontsize=fs(11),
                bbox=dict(facecolor='white', alpha=0.7, pad=2.0, edgecolor='none'))
    else:
        ax.text(0.5, 0.5, f"n<{BIN_MIN_N}", transform=ax.transAxes, ha='center',
                va='center', fontsize=fs(11), color='0.5')
    ax.set_xlim(*GLIM2); ax.set_ylim(*YLIM2)
    ax.set_xticks(TICKS); ax.set_yticks(TICKS)
    style_axes(ax)
    ax.set_xlabel(r"$\log M_\star$ true", fontsize=fs(12))
    if j == 0:
        ax.set_ylabel(r"$\log M_\star$ pred", fontsize=fs(12))
    else:
        ax.tick_params(labelleft=False)

fig.tight_layout()
handles, labels = top_panels[0].get_legend_handles_labels()
leg = top_panels[-1].legend(handles, labels, loc="upper right", ncol=1, frameon=True,
                            markerscale=1.6, handletextpad=0.3, labelspacing=0.25,
                            borderpad=0.25, prop={'size': fs(7)}, bbox_to_anchor=(0.98, 0.98))
leg.get_frame().set_edgecolor('none'); leg.get_frame().set_alpha(0.7); leg.set_zorder(5)

for j, ax in enumerate(top_panels):
    for x_edge, x_corner in ((EDGES[j], 0.0), (EDGES[j + 1], 1.0)):
        con = ConnectionPatch(xyA=(x_edge, 0.0), coordsA=axz.get_xaxis_transform(),
                              xyB=(x_corner, 1.0), coordsB=ax.transAxes,
                              color=BCOL[j], lw=2.4, alpha=1.0, zorder=0)
        con.set_clip_on(False); fig.add_artist(con)

fig.patch.set_alpha(0.0)
save(fig, "mass")
plt.show()

print("\n%-14s %6s %8s %8s" % ("z-bin", "n", "R2", "NMAD"))
for j in range(NB2):
    mk = _bin_mask2(j)
    if mk is None:
        print("%-14s %6s" % (f"({EDGES[j]:.1f},{EDGES[j+1]:.1f}]", "too few")); continue
    yv, pv = yva[mk], rpred[mk]; d = pv - yv
    print("%-14s %6d %8.3f %8.3f" % (f"({EDGES[j]:.1f},{EDGES[j+1]:.1f}]", len(yv),
          r2_score(yv, pv), 1.4826 * np.median(np.abs(d - np.median(d)))))